# LeetCode 1189 — Maximum Number of Balloons bằng Python

**Link đề:** https://leetcode.com/problems/maximum-number-of-balloons/
**Độ khó:** Easy · **Chủ đề:** Hash Table, String, Counting

```python
class Solution:
    def maxNumberOfBalloons(self, text: str) -> int:
        ...
```

## Mục tiêu học
1. Hiểu bài toán dạng **"từ kho nguyên liệu, ghép được tối đa bao nhiêu sản phẩm?"**.
2. Nắm kỹ thuật **đếm tần suất**: `dict`, `dict.get`, `collections.Counter`, mảng 26 ô với `ord`.
3. Hiểu vì sao đáp án là **min của các tỉ lệ** `có // cần` (giống "nút cổ chai" trong dây chuyền).
4. Đi từ cách mô phỏng (ghép từng quả bóng) → công thức `min` → tổng quát hoá cho từ bất kỳ (LeetCode 2287).
5. Trace từng bước, chứng minh tính đúng, test đối chiếu, làm bài tập TODO có đáp án ở cuối.

## 1. Đề bài và ví dụ

Cho chuỗi `text`. Ta muốn dùng các ký tự trong `text` để ghép thành **càng nhiều từ `"balloon"` càng tốt**.
Mỗi ký tự trong `text` chỉ được dùng **tối đa một lần**. Trả về số từ `"balloon"` tối đa ghép được.

| `text` | Kết quả | Vì sao |
|---|:---:|---|
| `"nlaebolko"` | `1` | có đủ b, a, l×2, o×2, n cho đúng 1 từ |
| `"loonbalxballpoon"` | `2` | đủ nguyên liệu cho 2 từ |
| `"leetcode"` | `0` | không có `b`, `a`, `n` |

**Constraints:** `1 <= text.length <= 10^4`; `text` chỉ gồm chữ cái tiếng Anh thường.

Ghi chú: bài này giống hệt bài **2287. Rearrange Characters to Make Target String** khi target là `"balloon"`.

**Điểm mấu chốt cần để ý ngay:** thứ tự ký tự trong `text` **không quan trọng** — ta được "lấy" ký tự ở đâu cũng được. Vậy chỉ có **số lượng** mỗi chữ là đáng quan tâm.

In [1]:
word = "balloon"
print("Từ cần ghép:", word, "| độ dài:", len(word))
for ch in word:
    print(ch, end=" ")
print()
print("Chữ 'l' xuất hiện", word.count("l"), "lần; chữ 'o' xuất hiện", word.count("o"), "lần")

Từ cần ghép: balloon | độ dài: 7
b a l l o o n 
Chữ 'l' xuất hiện 2 lần; chữ 'o' xuất hiện 2 lần


Quan sát: `"balloon"` có 7 ký tự nhưng chỉ 5 chữ khác nhau. Trong đó **`l` và `o` cần 2 lần** cho mỗi quả bóng.
Đây chính là cái bẫy của bài: nếu quên điều này, ta sẽ đếm sai.

| Chữ | Cần cho 1 "balloon" |
|:---:|:---:|
| b | 1 |
| a | 1 |
| l | 2 |
| o | 2 |
| n | 1 |

## 2. Nền tảng Python cần biết

### 2.1. Duyệt chuỗi và `str.count`

Chuỗi trong Python là dãy ký tự, duyệt được bằng `for ch in s`. Phương thức `s.count(x)` trả số lần `x` xuất hiện trong `s` (mỗi lần gọi phải duyệt cả chuỗi → `O(n)`).

In [2]:
text = "loonbalxballpoon"
for ch in "balon":
    print(f"text.count({ch!r}) = {text.count(ch)}")

text.count('b') = 2
text.count('a') = 2
text.count('l') = 4
text.count('o') = 4
text.count('n') = 2


### 2.2. `dict` để đếm tần suất

`dict` lưu cặp **key → value**. Ở đây key là ký tự, value là số lần đã gặp.

- `d[k] = v`: gán.
- `k in d`: kiểm tra có key chưa (trung bình `O(1)`).
- `d.get(k, 0)`: đọc value, nếu chưa có key thì trả `0` thay vì báo lỗi `KeyError`.

In [3]:
freq = {}
for ch in "hello":
    freq[ch] = freq.get(ch, 0) + 1   # chưa có -> 0 + 1; có rồi -> cộng thêm 1
print(freq)
print("'l' in freq ->", "l" in freq, "| freq.get('z', 0) ->", freq.get("z", 0))

{'h': 1, 'e': 1, 'l': 2, 'o': 1}
'l' in freq -> True | freq.get('z', 0) -> 0


Nếu viết `freq[ch] += 1` mà `ch` chưa có trong `freq`, Python báo `KeyError`. Đó là lý do ta dùng `get(ch, 0)`.

In [4]:
freq = {}
try:
    freq["a"] += 1
except KeyError as e:
    print("KeyError vì chưa có key:", e)

KeyError vì chưa có key: 'a'


### 2.3. `collections.Counter` — bộ đếm có sẵn

`Counter(iterable)` đếm luôn cho ta. Điểm hay: truy cập key **không có** trả `0`, không lỗi.
Counter còn hỗ trợ phép trừ `c1 - c2` (bỏ các giá trị ≤ 0) và so sánh "chứa" bằng `<=` (Python 3.10+).

In [5]:
from collections import Counter

cnt = Counter("loonbalxballpoon")
print(cnt)
print("cnt['l'] =", cnt["l"], "| cnt['z'] =", cnt["z"])   # 'z' không có -> 0
need = Counter("balloon")
print("need =", need)

Counter({'l': 4, 'o': 4, 'n': 2, 'b': 2, 'a': 2, 'x': 1, 'p': 1})
cnt['l'] = 4 | cnt['z'] = 0
need = Counter({'l': 2, 'o': 2, 'b': 1, 'a': 1, 'n': 1})


### 2.4. Phép chia nguyên `//` và hàm `min`

- `a // b`: chia lấy phần nguyên (làm tròn xuống). `5 // 2 = 2` — có 5 chữ `l`, mỗi quả bóng cần 2 → chỉ đủ cho 2 quả.
- `min(a, b, c, ...)` hoặc `min(iterable)`: lấy nhỏ nhất.

In [6]:
print("5 // 2 =", 5 // 2, "| 4 // 2 =", 4 // 2, "| 1 // 2 =", 1 // 2)
print("5 / 2 =", 5 / 2, "(chia thực — không dùng ở đây)")
print("min(3, 2, 7) =", min(3, 2, 7), "| min([4, 1, 9]) =", min([4, 1, 9]))

5 // 2 = 2 | 4 // 2 = 2 | 1 // 2 = 0
5 / 2 = 2.5 (chia thực — không dùng ở đây)
min(3, 2, 7) = 2 | min([4, 1, 9]) = 1


### 2.5. `ord` và mảng 26 ô

`ord(ch)` trả mã số của ký tự. Với chữ thường, `ord(ch) - ord('a')` cho vị trí `0..25`. Đây là cách đếm bằng list thay cho dict — nhanh, gọn khi bảng chữ cái cố định.

In [7]:
for ch in "abz":
    print(ch, "-> ord =", ord(ch), "-> index =", ord(ch) - ord("a"))
counts = [0] * 26
for ch in "balloon":
    counts[ord(ch) - ord("a")] += 1
print("counts['l'] =", counts[ord("l") - ord("a")], "| counts['o'] =", counts[ord("o") - ord("a")])

a -> ord = 97 -> index = 0
b -> ord = 98 -> index = 1
z -> ord = 122 -> index = 25
counts['l'] = 2 | counts['o'] = 2


## 3. Cách 1 — Mô phỏng: ghép từng quả bóng một

Cách trực giác nhất: giống như đi chợ. Ta có một "giỏ" ký tự. Mỗi vòng, thử **lấy ra** đủ 7 ký tự của `"balloon"`.
Lấy được → đếm thêm 1 quả. Thiếu bất kỳ chữ nào → dừng.

In [8]:
def balloons_simulate(text: str) -> int:
    stock = {}
    for ch in text:
        stock[ch] = stock.get(ch, 0) + 1

    made = 0
    while True:
        for ch in "balloon":                 # thử lấy 7 ký tự
            if stock.get(ch, 0) == 0:
                return made                  # thiếu -> dừng
            stock[ch] -= 1
        made += 1

for t in ["nlaebolko", "loonbalxballpoon", "leetcode"]:
    print(t, "->", balloons_simulate(t))

nlaebolko -> 1
loonbalxballpoon -> 2
leetcode -> 0


**Lưu ý nhỏ:** khi phát hiện thiếu chữ giữa chừng, ta đã lỡ trừ vài chữ của quả bóng dang dở. Không sao — vì ta trả `made` ngay, không dùng giỏ nữa.

### Chi phí của cách mô phỏng
- Đếm: `O(n)`.
- Vòng `while` chạy `answer + 1` lần, mỗi lần 7 bước. `answer ≤ n / 7` → tổng `O(n)`.

Cách này **đã đủ nhanh**, nhưng nó "làm thật" từng quả bóng. Ta có thể trả lời bằng **một phép tính** thay vì mô phỏng.

## 4. Nút thắt và insight — "nút cổ chai"

Hãy hình dung một xưởng ráp bóng: mỗi quả cần `1 b, 1 a, 2 l, 2 o, 1 n`.

- Có `b` đủ cho `cnt['b'] // 1` quả.
- Có `l` đủ cho `cnt['l'] // 2` quả.
- ... tương tự cho từng chữ.

Số quả ráp được bị giới hạn bởi **chữ khan hiếm nhất** (nút cổ chai):

```text
answer = min( cnt[b]//1, cnt[a]//1, cnt[l]//2, cnt[o]//2, cnt[n]//1 )
```

Ví dụ `"loonbalxballpoon"`:

In [9]:
from collections import Counter

text = "loonbalxballpoon"
cnt = Counter(text)
need = {"b": 1, "a": 1, "l": 2, "o": 2, "n": 1}
for ch, k in need.items():
    print(f"chữ {ch}: có {cnt[ch]}, cần {k}/quả -> đủ cho {cnt[ch] // k} quả")
print("answer = min(...) =", min(cnt[ch] // k for ch, k in need.items()))

chữ b: có 2, cần 1/quả -> đủ cho 2 quả
chữ a: có 2, cần 1/quả -> đủ cho 2 quả
chữ l: có 4, cần 2/quả -> đủ cho 2 quả
chữ o: có 4, cần 2/quả -> đủ cho 2 quả
chữ n: có 2, cần 1/quả -> đủ cho 2 quả
answer = min(...) = 2


`b`, `a`, `n` đều chỉ đủ cho 2 quả → đáp án 2, dù `l` (4) và `o` (4) cũng vừa khít 2 quả.

Câu `min(cnt[ch] // k for ch, k in need.items())` dùng **generator expression**: giống list comprehension nhưng không tạo list, đưa thẳng từng giá trị vào `min`.

In [10]:
squares = [x * x for x in range(5)]          # list comprehension -> tạo list
print(squares)
print(min(x * x for x in range(1, 5)))       # generator -> không cần dấu []
d = {"x": 3, "y": 7}
for key, value in d.items():                 # .items() trả từng cặp (key, value)
    print(key, value)

[0, 1, 4, 9, 16]
1
x 3
y 7


## 5. Lời giải chính — giảng từng đoạn rồi ghép

**Bước 1 — Đếm `text`.** Dùng `Counter(text)`.

**Bước 2 — Đếm nhu cầu của từ đích.** `Counter("balloon")` tự cho `{'l': 2, 'o': 2, 'b': 1, 'a': 1, 'n': 1}` — không cần gõ tay, tránh sai.

**Bước 3 — Với mỗi chữ cần, tính số quả nó cho phép**: `cnt[ch] // need[ch]`.

**Bước 4 — Lấy `min`.**

In [11]:
from collections import Counter

text = "nlaebolko"
cnt = Counter(text)                 # bước 1
need = Counter("balloon")           # bước 2
ratios = {ch: cnt[ch] // k for ch, k in need.items()}   # bước 3 (dict comprehension)
print("cnt   =", dict(cnt))
print("need  =", dict(need))
print("ratios=", ratios)
print("answer=", min(ratios.values()))  # bước 4

cnt   = {'n': 1, 'l': 2, 'a': 1, 'e': 1, 'b': 1, 'o': 2, 'k': 1}
need  = {'b': 1, 'a': 1, 'l': 2, 'o': 2, 'n': 1}
ratios= {'b': 1, 'a': 1, 'l': 1, 'o': 1, 'n': 1}
answer= 1


Ghép lại thành hàm. Ta viết tổng quát cho **từ đích bất kỳ** `target` để dùng lại sau (bài 2287):

In [12]:
from collections import Counter

def max_words(text: str, target: str) -> int:
    have = Counter(text)
    need = Counter(target)
    return min(have[ch] // k for ch, k in need.items())

def balloons_counter(text: str) -> int:
    return max_words(text, "balloon")

for t in ["nlaebolko", "loonbalxballpoon", "leetcode"]:
    print(t, "->", balloons_counter(t))

nlaebolko -> 1
loonbalxballpoon -> 2
leetcode -> 0


### Cách 3 — Mảng 26 ô (không dùng thư viện)

Phỏng vấn đôi khi yêu cầu không dùng `Counter`. Ta đếm bằng list 26 ô:

In [13]:
def balloons_array(text: str) -> int:
    counts = [0] * 26
    for ch in text:
        counts[ord(ch) - ord("a")] += 1

    def c(ch):
        return counts[ord(ch) - ord("a")]

    return min(c("b"), c("a"), c("l") // 2, c("o") // 2, c("n"))

for t in ["nlaebolko", "loonbalxballpoon", "leetcode"]:
    print(t, "->", balloons_array(t))

nlaebolko -> 1
loonbalxballpoon -> 2
leetcode -> 0


### Cách 4 — Một dòng với `str.count`

Vì chỉ có 5 chữ cần quan tâm, gọi `count` 5 lần (mỗi lần `O(n)`) vẫn là `O(n)` tổng:

In [14]:
def balloons_count(text: str) -> int:
    return min(text.count("b"), text.count("a"), text.count("l") // 2,
               text.count("o") // 2, text.count("n"))

for t in ["nlaebolko", "loonbalxballpoon", "leetcode"]:
    print(t, "->", balloons_count(t))

nlaebolko -> 1
loonbalxballpoon -> 2
leetcode -> 0


## 6. TRACE từng bước

### 6.1. Trace cách đếm + min trên `"loonbalxballpoon"`

| Bước | Ký tự đọc | Thay đổi | `b a l o n` sau bước |
|:---:|:---:|---|---|
| 1 | l | l: 0→1 | 0 0 1 0 0 |
| 2 | o | o: 0→1 | 0 0 1 1 0 |
| ... | ... | ... | ... |
| 16 | n | n: 1→2 | 2 2 4 4 2 |

Cuối cùng: tỉ lệ `b=2, a=2, l=4//2=2, o=4//2=2, n=2` → `min = 2`.

Hàm dưới in chính xác từng dòng (bỏ qua chữ không liên quan như `x`, `p`):

In [15]:
def trace_balloons(text: str) -> int:
    need = {"b": 1, "a": 1, "l": 2, "o": 2, "n": 1}
    have = {ch: 0 for ch in need}
    for i, ch in enumerate(text):
        if ch in have:
            have[ch] += 1
            state = " ".join(f"{k}={have[k]}" for k in need)
            print(f"i={i:2d} ch={ch!r}: cộng 1 -> {state}")
        else:
            print(f"i={i:2d} ch={ch!r}: không thuộc 'balloon', bỏ qua")
    ratios = {ch: have[ch] // need[ch] for ch in need}
    print("tỉ lệ có//cần:", ratios)
    ans = min(ratios.values())
    bottleneck = [ch for ch in need if ratios[ch] == ans]
    print("answer =", ans, "| nút cổ chai:", bottleneck)
    return ans

assert trace_balloons("loonbalxballpoon") == 2

i= 0 ch='l': cộng 1 -> b=0 a=0 l=1 o=0 n=0
i= 1 ch='o': cộng 1 -> b=0 a=0 l=1 o=1 n=0
i= 2 ch='o': cộng 1 -> b=0 a=0 l=1 o=2 n=0
i= 3 ch='n': cộng 1 -> b=0 a=0 l=1 o=2 n=1
i= 4 ch='b': cộng 1 -> b=1 a=0 l=1 o=2 n=1
i= 5 ch='a': cộng 1 -> b=1 a=1 l=1 o=2 n=1
i= 6 ch='l': cộng 1 -> b=1 a=1 l=2 o=2 n=1
i= 7 ch='x': không thuộc 'balloon', bỏ qua
i= 8 ch='b': cộng 1 -> b=2 a=1 l=2 o=2 n=1
i= 9 ch='a': cộng 1 -> b=2 a=2 l=2 o=2 n=1
i=10 ch='l': cộng 1 -> b=2 a=2 l=3 o=2 n=1
i=11 ch='l': cộng 1 -> b=2 a=2 l=4 o=2 n=1
i=12 ch='p': không thuộc 'balloon', bỏ qua
i=13 ch='o': cộng 1 -> b=2 a=2 l=4 o=3 n=1
i=14 ch='o': cộng 1 -> b=2 a=2 l=4 o=4 n=1
i=15 ch='n': cộng 1 -> b=2 a=2 l=4 o=4 n=2
tỉ lệ có//cần: {'b': 2, 'a': 2, 'l': 2, 'o': 2, 'n': 2}
answer = 2 | nút cổ chai: ['b', 'a', 'l', 'o', 'n']


`enumerate(text)` trả từng cặp `(chỉ số, ký tự)` — tiện khi muốn in vị trí.

### 6.2. Trace cách mô phỏng trên `"nlaebolko"`

Ở đây thấy rõ: quả thứ 1 ghép thành công, quả thứ 2 thất bại ngay ở chữ `b`.

In [16]:
def trace_simulate(text: str) -> int:
    stock = {}
    for ch in text:
        stock[ch] = stock.get(ch, 0) + 1
    print("giỏ ban đầu:", stock)
    made = 0
    while True:
        for ch in "balloon":
            if stock.get(ch, 0) == 0:
                print(f"  quả #{made + 1}: thiếu {ch!r} -> dừng, trả {made}")
                return made
            stock[ch] -= 1
        made += 1
        print(f"  quả #{made} OK, giỏ còn:", {k: v for k, v in stock.items() if v})

assert trace_simulate("nlaebolko") == 1

giỏ ban đầu: {'n': 1, 'l': 2, 'a': 1, 'e': 1, 'b': 1, 'o': 2, 'k': 1}
  quả #1 OK, giỏ còn: {'e': 1, 'k': 1}
  quả #2: thiếu 'b' -> dừng, trả 1


## 7. Code nộp LeetCode

### 7.1. Bản chính — Counter + min (gọn, tổng quát)

In [17]:
from collections import Counter

class Solution:
    def maxNumberOfBalloons(self, text: str) -> int:
        have = Counter(text)
        need = Counter("balloon")
        return min(have[ch] // k for ch, k in need.items())

sol = Solution()
print(sol.maxNumberOfBalloons("nlaebolko"), sol.maxNumberOfBalloons("loonbalxballpoon"), sol.maxNumberOfBalloons("leetcode"))

1 2 0


### 7.2. Bản không thư viện — mảng 26 ô

In [18]:
class SolutionArray:
    def maxNumberOfBalloons(self, text: str) -> int:
        counts = [0] * 26
        for ch in text:
            counts[ord(ch) - 97] += 1          # 97 == ord('a')
        b, a, l, o, n = (counts[ord(c) - 97] for c in "balon")
        return min(b, a, l // 2, o // 2, n)

s2 = SolutionArray()
assert [s2.maxNumberOfBalloons(t) for t in ["nlaebolko", "loonbalxballpoon", "leetcode"]] == [1, 2, 0]
print("SolutionArray OK")

SolutionArray OK


### 7.3. Bản mô phỏng (đúng nhưng dài hơn)

In [19]:
class SolutionSimulate:
    def maxNumberOfBalloons(self, text: str) -> int:
        stock = {}
        for ch in text:
            stock[ch] = stock.get(ch, 0) + 1
        made = 0
        while True:
            for ch in "balloon":
                if stock.get(ch, 0) == 0:
                    return made
                stock[ch] -= 1
            made += 1

assert SolutionSimulate().maxNumberOfBalloons("loonbalxballpoon") == 2
print("SolutionSimulate OK")

SolutionSimulate OK


## 8. Vì sao đúng?

Gọi `k` = số quả bóng ghép được. Để ghép `k` quả cần `k` chữ `b`, `k` chữ `a`, `2k` chữ `l`, `2k` chữ `o`, `k` chữ `n`.

- **Cận trên:** ghép được `k` quả ⟹ `have[l] ≥ 2k` ⟹ `k ≤ have[l] // 2` (vì `k` nguyên). Tương tự cho mọi chữ. Vậy `k ≤ min(...)`.
- **Đạt được:** đặt `m = min(...)`. Với mọi chữ `ch`, `have[ch] // need[ch] ≥ m` ⟹ `have[ch] ≥ m·need[ch]` → đủ nguyên liệu cho `m` quả. Vì thứ tự ký tự không quan trọng, cứ lấy bất kỳ.

Hai chiều gộp lại: `k_max = min(have[ch] // need[ch])`. ∎

**Bất biến của cách mô phỏng:** sau mỗi vòng `while` thành công, `stock` = số chữ ban đầu trừ đúng `made` lần nhu cầu của `"balloon"`. Vòng dừng đúng lúc lần đầu tiên một chữ không đủ — chính là `min` ở trên.

## 9. Độ phức tạp và so sánh

Gọi `n = len(text)`, `σ` = số chữ khác nhau của từ đích (ở đây 5).

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| Mô phỏng | `O(n)` | `O(1)` (≤ 26 key) | dễ nghĩ, dài |
| Counter + min | `O(n)` | `O(1)` | gọn, tổng quát cho target bất kỳ |
| Mảng 26 ô | `O(n)` | `O(1)` | không thư viện, nhanh |
| `str.count` × 5 | `O(5n)=O(n)` | `O(1)` | 1 dòng, nhưng đọc chuỗi 5 lần |

Bộ nhớ coi là `O(1)` vì bảng chữ cái cố định 26 ký tự (không lớn theo `n`).

## 10. Bộ test

Gồm: mẫu đề, biên (1 ký tự, chữ đúng nhưng thiếu cặp `l`/`o`), phản ví dụ cho lỗi "quên chia 2", và đối chiếu 4 cách trên nhiều input ngẫu nhiên.

In [20]:
impls = {
    "simulate": balloons_simulate,
    "counter": balloons_counter,
    "array": balloons_array,
    "count": balloons_count,
    "Solution": Solution().maxNumberOfBalloons,
}
cases = [
    ("nlaebolko", 1),
    ("loonbalxballpoon", 2),
    ("leetcode", 0),
    ("b", 0),                      # biên: 1 ký tự
    ("balon", 0),                  # có đủ loại chữ nhưng thiếu l, o thứ 2
    ("balloon", 1),
    ("balloonballoon", 2),
    ("bbbbbaaaaallllloooon", 1),   # n là nút cổ chai
    ("bbaalllloooonn", 2),
    ("balllllllllooooooooon", 1),  # thừa l,o nhưng b/a/n chỉ 1
]
for text, expected in cases:
    for name, f in impls.items():
        got = f(text)
        assert got == expected, (name, text, got, expected)
    print(f"{text!r:26} -> {expected}  PASS (cả {len(impls)} cách)")

'nlaebolko'                -> 1  PASS (cả 5 cách)
'loonbalxballpoon'         -> 2  PASS (cả 5 cách)
'leetcode'                 -> 0  PASS (cả 5 cách)
'b'                        -> 0  PASS (cả 5 cách)
'balon'                    -> 0  PASS (cả 5 cách)
'balloon'                  -> 1  PASS (cả 5 cách)
'balloonballoon'           -> 2  PASS (cả 5 cách)
'bbbbbaaaaallllloooon'     -> 1  PASS (cả 5 cách)
'bbaalllloooonn'           -> 2  PASS (cả 5 cách)
'balllllllllooooooooon'    -> 1  PASS (cả 5 cách)


In [21]:
import random

rng = random.Random(1189)
for _ in range(500):
    n = rng.randint(1, 40)
    text = "".join(rng.choice("balonxyz") for _ in range(n))
    results = {name: f(text) for name, f in impls.items()}
    assert len(set(results.values())) == 1, (text, results)
print("500 test ngẫu nhiên: tất cả các cách cho cùng kết quả")

big = "balloon" * 1428 + "ba"      # ~10^4 ký tự
print("len =", len(big), "->", Solution().maxNumberOfBalloons(big))
assert Solution().maxNumberOfBalloons(big) == 1428

500 test ngẫu nhiên: tất cả các cách cho cùng kết quả
len = 9998 -> 1428


## 11. Lỗi người mới hay gặp

### Lỗi 1: Quên `l` và `o` cần 2 lần

```python
return min(cnt['b'], cnt['a'], cnt['l'], cnt['o'], cnt['n'])   # SAI
```

Vì sao sai: với `"balon"` mỗi chữ có 1 → trả 1, nhưng thực tế không ghép nổi `"balloon"`.

In [22]:
from collections import Counter

def wrong_no_div(text):
    c = Counter(text)
    return min(c["b"], c["a"], c["l"], c["o"], c["n"])

print("wrong('balon') =", wrong_no_div("balon"), "| đúng =", balloons_counter("balon"))

wrong('balon') = 1 | đúng = 0


### Lỗi 2: Dùng `set("balloon")` rồi chia cho số lần trong `text`

`set` làm mất thông tin "cần 2 lần". Luôn dùng `Counter(target)` để giữ số lượng.

### Lỗi 3: Dùng `/` thay vì `//`

`3 / 2 = 1.5` → trả về số thực, sai kiểu và sai giá trị (không ghép được nửa quả bóng).

In [23]:
from collections import Counter

def wrong_float(text):
    c = Counter(text)
    return min(c["b"], c["a"], c["l"] / 2, c["o"] / 2, c["n"])

print("wrong_float('bbaalllooonn') =", wrong_float("bbaalllooonn"), "| đúng =", balloons_counter("bbaalllooonn"))

wrong_float('bbaalllooonn') = 1.5 | đúng = 1


### Lỗi 4: Dùng `dict` thường và truy cập key không có

```python
d = {}
for ch in text: d[ch] = d.get(ch, 0) + 1
return min(d['b'], ...)      # KeyError nếu text không có 'b'
```

Sửa: dùng `d.get('b', 0)` hoặc `Counter` (trả 0 cho key thiếu).

In [24]:
def wrong_keyerror(text):
    d = {}
    for ch in text:
        d[ch] = d.get(ch, 0) + 1
    return min(d["b"], d["a"], d["l"] // 2, d["o"] // 2, d["n"])

try:
    wrong_keyerror("leetcode")
except KeyError as e:
    print("KeyError:", e, "-> phải dùng .get(ch, 0) hoặc Counter")

KeyError: 'b' -> phải dùng .get(ch, 0) hoặc Counter


## 12. Mẹo nhận dạng pattern

- Đề nói **"dùng mỗi ký tự tối đa một lần"**, **"ghép/tạo được bao nhiêu"** và **thứ tự không quan trọng** → nghĩ ngay tới **đếm tần suất**.
- Câu hỏi "tối đa bao nhiêu bộ" với mỗi bộ cần một lượng cố định của từng loại → **`min(có // cần)`**.
- Bảng chữ cái nhỏ cố định (26 chữ) → mảng 26 ô hoặc `Counter`, bộ nhớ `O(1)`.

Bài cùng họ: 383 Ransom Note (hỏi có ghép được **1** lần không → `need <= have`), 2287 Rearrange Characters to Make Target String, 1160 Find Words That Can Be Formed by Characters.

## 13. Biến thể

### 13.1. Từ đích bất kỳ (LeetCode 2287)

Hàm `max_words(text, target)` ở mục 5 đã giải luôn bài này.

In [25]:
print(max_words("ilovecodingonleetcode", "code"))   # 2
print(max_words("abcba", "abc"))                    # 1
print(max_words("abbaccaddaeea", "aaaaa"))          # 1

2
1
1


### 13.2. Ransom Note — chỉ hỏi có ghép được ít nhất 1 lần

Đó chính là `max_words(...) >= 1`. Hoặc dùng phép so sánh Counter: `Counter(note) <= Counter(magazine)` (Python 3.10+). Ở đây viết bằng vòng lặp cho chắc chắn mọi phiên bản:

In [26]:
from collections import Counter

def can_construct(note: str, magazine: str) -> bool:
    have = Counter(magazine)
    return all(have[ch] >= k for ch, k in Counter(note).items())

print(can_construct("aa", "aab"), can_construct("aa", "ab"))

True False


`all(iterable)` trả `True` nếu mọi phần tử đúng (và `True` với iterable rỗng).

### 13.3. Còn thừa bao nhiêu ký tự sau khi ghép tối đa?

Sau khi ghép `k` quả, phần thừa = `have - k * need`. Có thể dùng để trả lời "còn thiếu gì để ghép thêm 1 quả".

In [27]:
from collections import Counter

def missing_for_next(text: str, target: str = "balloon") -> dict:
    have, need = Counter(text), Counter(target)
    k = min(have[ch] // c for ch, c in need.items())
    return {ch: (k + 1) * c - have[ch] for ch, c in need.items() if have[ch] < (k + 1) * c}

print(missing_for_next("loonbalxballpoon"))   # cần thêm b, a, n và 1 cặp l, o nữa

{'b': 1, 'a': 1, 'l': 2, 'o': 2, 'n': 1}


## 14. Bài tập (tự làm — các hàm TODO KHÔNG được gọi khi chạy notebook)

**Bài 1 (tính tay).** Với `text = "bbaallloooonnx"`, lập bảng `có / cần / tỉ lệ` cho 5 chữ và cho biết đáp án, nút cổ chai.

**Bài 2 (dự đoán output).** Không chạy, đoán: `Counter("balloon")["l"]`, `Counter("abc")["z"]`, `7 // 2`, `min(3, 1 // 2, 5)`.

**Bài 3 (điền TODO).** Hoàn thiện hàm đếm bằng `dict` + `get`, rồi trả về số `"balloon"`.

In [28]:
def exercise_balloons_dict(text: str) -> int:
    freq = {}
    # TODO 1: duyệt từng ký tự, tăng freq[ch] bằng freq.get(ch, 0) + 1
    # TODO 2: tính min của b, a, l//2, o//2, n (nhớ dùng .get(..., 0))
    pass

# Tự làm xong rồi mới mở test:
# assert exercise_balloons_dict("loonbalxballpoon") == 2
# assert exercise_balloons_dict("leetcode") == 0

**Bài 4 (viết lại).** Viết `exercise_max_words(text, target)` tổng quát cho mọi `target`, **không** dùng `Counter` (dùng mảng 26 ô).

In [29]:
def exercise_max_words(text: str, target: str) -> int:
    # TODO: đếm text vào have[26], target vào need[26]
    # TODO: chỉ xét chữ có need > 0, lấy min(have // need)
    pass

# assert exercise_max_words("ilovecodingonleetcode", "code") == 2
# assert exercise_max_words("abcba", "abc") == 1

**Bài 5 (biến thể).** Cho danh sách từ `words` và chuỗi `chars`. Một từ "tốt" nếu ghép được từ `chars` (mỗi ký tự dùng ≤ 1 lần, **mỗi từ xét riêng**). Trả tổng độ dài các từ tốt (LeetCode 1160).

In [30]:
def exercise_count_characters(words: list[str], chars: str) -> int:
    # TODO: với mỗi word, kiểm tra Counter(word) "nằm trong" Counter(chars)
    pass

# assert exercise_count_characters(["cat", "bt", "hat", "tree"], "atach") == 6
# assert exercise_count_characters(["hello", "world", "leetcode"], "welldonehoneyr") == 10

**Bài 6 (mini challenge).** Ghép tối đa **cặp** từ `"balloon"` và `"ball"` sao cho số `"balloon"` = số `"ball"` (mỗi cặp gồm 1 từ mỗi loại). Trả số cặp tối đa. Gợi ý: một cặp tương đương với một "từ gộp".

In [31]:
def exercise_balloon_ball_pairs(text: str) -> int:
    # TODO
    pass

# assert exercise_balloon_ball_pairs("balloonball") == 1
# assert exercise_balloon_ball_pairs("balloonballoonball") == 1

## 15. Gợi ý 3 mức

| Bài | Mức 1 — hướng nghĩ | Mức 2 — cấu trúc | Mức 3 — gần code |
|---|---|---|---|
| 1 | Đếm từng chữ trong `text` | `có // cần` | với `l`: `3 // 2 = ?` — chữ nào cho tỉ lệ nhỏ nhất? |
| 2 | Counter trả 0 khi thiếu key | `//` làm tròn xuống | `1 // 2` là bao nhiêu? rồi mới lấy `min` |
| 3 | Đếm rồi lấy nút cổ chai | `freq.get(ch, 0)` | `min(g('b'), g('a'), g('l')//2, ...)` |
| 4 | Như Counter nhưng 26 ô | `ord(ch) - 97` | `min(have[i] // need[i] for i in range(26) if need[i])` |
| 5 | Mỗi từ là một Ransom Note | `all(have[c] >= k ...)` | cộng `len(word)` nếu tốt |
| 6 | Cặp = target `"balloon" + "ball"` | dùng lại `max_words` | `max_words(text, "balloonball")` |

## 16. Đáp án tham khảo (chỉ xem sau khi tự làm)

**Bài 1:** `b=2, a=2, l=3, o=4, n=2` → tỉ lệ `2, 2, 1, 2, 2` → đáp án **1**, nút cổ chai `l`.

**Bài 2:** `2`, `0`, `3`, `0`.

In [32]:
from collections import Counter

def answer_balloons_dict(text: str) -> int:
    freq = {}
    for ch in text:
        freq[ch] = freq.get(ch, 0) + 1
    g = lambda c: freq.get(c, 0)
    return min(g("b"), g("a"), g("l") // 2, g("o") // 2, g("n"))

def answer_max_words(text: str, target: str) -> int:
    have, need = [0] * 26, [0] * 26
    for ch in text:
        have[ord(ch) - 97] += 1
    for ch in target:
        need[ord(ch) - 97] += 1
    return min(have[i] // need[i] for i in range(26) if need[i])

def answer_count_characters(words: list[str], chars: str) -> int:
    have = Counter(chars)
    total = 0
    for w in words:
        if all(have[c] >= k for c, k in Counter(w).items()):
            total += len(w)
    return total

def answer_balloon_ball_pairs(text: str) -> int:
    return answer_max_words(text, "balloon" + "ball")

# Bài 1, 2
assert balloons_counter("bbaallloooonnx") == 1
assert (Counter("balloon")["l"], Counter("abc")["z"], 7 // 2, min(3, 1 // 2, 5)) == (2, 0, 3, 0)
# Bài 3
assert answer_balloons_dict("loonbalxballpoon") == 2
assert answer_balloons_dict("leetcode") == 0
# Bài 4
assert answer_max_words("ilovecodingonleetcode", "code") == 2
assert answer_max_words("abcba", "abc") == 1
# Bài 5
assert answer_count_characters(["cat", "bt", "hat", "tree"], "atach") == 6
assert answer_count_characters(["hello", "world", "leetcode"], "welldonehoneyr") == 10
# Bài 6
assert answer_balloon_ball_pairs("balloonball") == 1
assert answer_balloon_ball_pairs("balloonballoonball") == 1
print("Tất cả đáp án PASS")

Tất cả đáp án PASS


## 17. Template nhớ nhanh và checklist submit

```python
from collections import Counter

def max_copies(text, target):
    have, need = Counter(text), Counter(target)
    return min(have[c] // k for c, k in need.items())
```

**Câu thần chú:** *"Thứ tự không quan trọng → đếm. Hỏi bao nhiêu bộ → min(có // cần)."*

### Checklist trước khi submit
- [ ] Đã chia cho **2** với `l` và `o` (hoặc dùng `Counter("balloon")` để khỏi nhớ tay)?
- [ ] Dùng `//` (chia nguyên) chứ không phải `/`?
- [ ] Không bị `KeyError` khi `text` thiếu một chữ (dùng `Counter` hoặc `.get(c, 0)`)?
- [ ] Trả `0` khi không ghép được quả nào?
- [ ] Tên method đúng `maxNumberOfBalloons`, nhận `text: str`, trả `int`?